In [ ]:
%run /Workspace/consolidated_pipeline/1_setup/utilities

In [ ]:
# ---> GOLD PROCESSING <---
from pypark.sql import functions as F
from delta.tables import DeltaTable

dbutils.widgets.text('data_source', 'customers', 'Data Source')
dbutils.widgets.text('catalog', 'fmcg', 'Catalog')

data_source = dbutils.widgets.get('data_source')
catalog = dbutils.widgets.get('catalog')

df_silver = spark.sql(f'SELECT * FROM {catalog}.{silver_schema}.{data_source};')

df_gold = df_silver.select('customer_id', 'customer_name', 'city', 'customer', 'market', 'platform', 'channel')

In [ ]:
df_gold.write\
    .format('delta') \
    .options('delta.enableChangeDataFeed', 'true') \
    .mode('overwrite') \
    .saveAsTable(f'{catalog}.{gold_schema}.db_dim_{data_source}')

In [ ]:
# ---> MERGIN DATA SOURCE WITH PARENT <---
# ---> UPSERT APPROACH <---
delta_table = DeltaTable.forName(spark, 'fmcg.gold.dim_customers')
df_child_customers = spark.table('fmcg.gold.db_dim_customers').select(
    'customer',
    'market',
    'platform',
    'channel'
)

delta_table.alias('target').merge(
    source=df_child_customers.alias('source'),
    condition='target.customer_code = source.customer_code'
).whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()